## Регрессия для SI

### Импортируем библиотеки

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

### Загружаем данные

In [9]:
df_ml = pd.read_csv('data_classic_ML.csv')

Удалим метрики при обучении т.к. если оставить IC50 и CC50 в признаках, модель мгновенно «догадается» вычесть из логарифма безопасности логарифм индекса селективности. Алгоритм покажет идеальную точность, но модель будет абсолютно бесполезна на практике, так как для новых (еще не синтезированных) молекул значения CC50 и SI заранее неизвестны. Модель должна опираться строго на топологию и свойства самого химического элемента.

In [10]:
target_cols = ['log_IC50', 'log_CC50', 'log_SI']
X = df_ml.drop(columns=[col for col in target_cols if col in df_ml.columns])
y = df_ml['log_SI']

Разбиение на обучающую и тестовую выборки (70% на 30%)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Масштабирование дескрипторов через алгортм RobustScaler

In [12]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Модель 1: Линейная регрессия

In [13]:
ridge_model = Ridge(alpha=5.0)
ridge_model.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_model.predict(X_test_scaled)

r2_ridge = r2_score(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

print(f"Коэффициент детерминации (R²): {r2_ridge:.4f}")
print(f"Среднеквадратичная ошибка (MSE): {mse_ridge:.4f}")
print(f"Средняя абсолютная ошибка (MAE): {mae_ridge:.4f}")

Коэффициент детерминации (R²): 0.1562
Среднеквадратичная ошибка (MSE): 0.5044
Средняя абсолютная ошибка (MAE): 0.5353


### Модель 2: Случайный лес

Подбор гиперпараметров

In [14]:
param_grid = {
    'n_estimators': [300, 500, 700],
    'max_depth': [12, 15, 18],
    'min_samples_split': [2, 4, 6],
    'max_features': ['sqrt', 'log2', 0.3]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1), 
    param_grid=param_grid, 
    cv=5, 
    scoring='r2', 
    n_jobs=-1, 
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"Лучшие подобранные параметры: {grid_search.best_params_}")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Лучшие подобранные параметры: {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_split': 6, 'n_estimators': 700}


Обучение модели

In [15]:
rf_model = RandomForestRegressor(
    n_estimators=700,
    max_depth=12,
    min_samples_split=6,
    max_features='sqrt',
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(f"Коэффициент детерминации (R²): {r2_rf:.4f}")
print(f"Среднеквадратичная ошибка (MSE): {mse_rf:.4f}")
print(f"Средняя абсолютная ошибка (MAE): {mae_rf:.4f}")

Коэффициент детерминации (R²): 0.2619
Среднеквадратичная ошибка (MSE): 0.4413
Средняя абсолютная ошибка (MAE): 0.4947


### Модель 3: Гистограммный градиентный бустинг

Подбор гиперпараметров

In [16]:
param_grid = {
    'max_iter': [100, 300, 500],
    'learning_rate': [0.02, 0.04, 0.08],
    'max_depth': [4, 6, 8],
    'l2_regularization': [0.1, 1.0, 5.0]
}
grid_search_hgb = GridSearchCV(
    estimator=HistGradientBoostingRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search_hgb.fit(X_train_scaled, y_train)

print(f"Лучшие подобранные параметры: {grid_search_hgb.best_params_}")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Лучшие подобранные параметры: {'l2_regularization': 5.0, 'learning_rate': 0.08, 'max_depth': 4, 'max_iter': 100}


In [17]:
hgb_model = HistGradientBoostingRegressor(
    max_iter=100, 
    learning_rate=0.08, 
    max_depth=4, 
    l2_regularization=5.0, 
    random_state=42
)
hgb_model.fit(X_train_scaled, y_train)
y_pred_hgb = hgb_model.predict(X_test_scaled)

r2_hgb = r2_score(y_test, y_pred_hgb)
mse_hgb = mean_squared_error(y_test, y_pred_hgb)
mae_hgb = mean_absolute_error(y_test, y_pred_hgb)

print(f"Коэффициент детерминации (R²): {r2_hgb:.4f}")
print(f"Среднеквадратичная ошибка (MSE): {mse_hgb:.4f}")
print(f"Средняя абсолютная ошибка (MAE): {mae_hgb:.4f}")

Коэффициент детерминации (R²): 0.2093
Среднеквадратичная ошибка (MSE): 0.4727
Средняя абсолютная ошибка (MAE): 0.5158


### Анализ результата

In [18]:
summary_data = {
    'Модель': ['Линейная регрессия', 'Случайный лес', 'Градиентный бустинг'],
    'R²': [r2_ridge, r2_rf, r2_hgb],
    'MSE': [mse_ridge, mse_rf, mse_hgb],
    'MAE': [mae_ridge, mae_rf, mae_hgb]
}
df_summary = pd.DataFrame(summary_data).round(4)
print(df_summary.to_string(index=False))

             Модель     R²    MSE    MAE
 Линейная регрессия 0.1562 0.5044 0.5353
      Случайный лес 0.2619 0.4413 0.4947
Градиентный бустинг 0.2093 0.4727 0.5158


Биологическая и математическая природа? индекс селективности - это производная величина, рассчитываемая как отношение безопасности к эффективности SI = CC50 / IC50. Поскольку механизмы подавления патогена и механизмы токсичности для здоровых клеток регулируются абсолютно разными биологическими путями, их математическое объединение удваивает уровень «шума» в данных.Сложность для моделей, алгоритмам приходится искать в химической структуре компромисс - признаки, которые одновременно снижают IC50 и повышают CC50. Из-за этого выявить чистую закономерность по базовым одномерным дескрипторам становится экстремально трудно, что и привело к падению коэффициента детерминации до 26% у лучшей модели Случайный лес.

Рекомендации по улучшению моделей:   
- Перевести задачу в формат бинарной классификации, разделив молекулы на «перспективные» (класс 1) и «неперспективные» (класс 0) на основе жесткого фармакологического критерия терапевтического окна, например SI>8.